# DPED: Deep Pathway for Edge Detection

**Bio-Inspired Deep Learning Model**

**Bio-Inspiration**: Hierarchical visual pathway simulation  
**Deep Learning Enhancement**: Swin Transformer integration  
**Improvement Area**: Precision and efficiency

Combines biological visual hierarchy with modern transformers.

In [ ]:
from pathlib import Path
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'opencv-python', 'numpy', 'tqdm', 'scikit-learn'], check=False)

import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, cv2
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score

PROJECT_ROOT = Path('..')
OUTPUT_DIR = PROJECT_ROOT / 'bio DL' / 'outputs' / 'DPED'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## DPED with Swin-like Architecture

In [ ]:
class WindowAttention(nn.Module):
    """Simplified window attention"""
    def __init__(self, dim):
        super().__init__()
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, C).permute(2, 0, 1, 3)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) / (C ** 0.5)
        return self.proj((attn.softmax(dim=-1) @ v))

class DPED(nn.Module):
    """DPED with hierarchical pathway and transformer"""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.attn = WindowAttention(128)
        self.conv3 = nn.Conv2d(128, 256, 3, padding=1)
        self.edge_out = nn.Conv2d(256, 1, 1)
    def forward(self, x):
        h, w = x.shape[2:]
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(F.max_pool2d(x, 2)))
        # Apply attention
        B, C, H, W = x.shape
        x_flat = x.flatten(2).transpose(1, 2)
        x_attn = self.attn(x_flat).transpose(1, 2).reshape(B, C, H, W)
        x = x + x_attn
        x = F.relu(self.conv3(F.max_pool2d(x, 2)))
        edge = F.interpolate(self.edge_out(x), (h, w), mode='bilinear', align_corners=False)
        return torch.sigmoid(edge)

model = DPED().to(DEVICE).eval()
print(f"✓ DPED: {sum(p.numel() for p in model.parameters()):,} params")

In [ ]:
class EdgeDataset(Dataset):
    def __init__(self, root, split='test'):
        self.img_dir, self.gt_dir = root / split / 'images', root / split / 'edges'
        self.images = sorted(list(self.img_dir.glob('*.jpg')) + list(self.img_dir.glob('*.png')))[:20]
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img_path = self.images[idx]
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        gt = cv2.imread(str(self.gt_dir / img_path.name.replace('.jpg', '.png')), 0).astype(np.float32) / 255.0 if (self.gt_dir / img_path.name.replace('.jpg', '.png')).exists() else np.zeros(img.shape[:2], dtype=np.float32)
        return torch.from_numpy(img.transpose(2, 0, 1)), torch.from_numpy(gt), img_path.name

loader = DataLoader(EdgeDataset(PROJECT_ROOT / 'datasets' / 'HED_Small', 'test'), batch_size=1)
preds, gts = [], []
with torch.no_grad():
    for imgs, gt, _ in tqdm(loader):
        preds.extend([model(imgs.to(DEVICE))[i,0].cpu().numpy() for i in range(imgs.shape[0])])
        gts.extend([gt[i].cpu().numpy() for i in range(gt.shape[0])])

def metrics(preds, labels):
    t, ois, ap, al = np.linspace(0.05, 0.95, 30), [], [], []
    for p, l in zip(preds, labels):
        l = cv2.dilate((l > 0.5).astype(np.float32), np.ones((3,3))).flatten()
        p = cv2.GaussianBlur(p, (3,3), 0).flatten()
        ap.append(p); al.append(l)
        ois.append(max([2*np.sum((p>=th)*l)/(2*np.sum((p>=th)*l)+np.sum((p>=th)*(1-l))+np.sum((p<th)*l)+1e-8) for th in t]))
    fp, fl = np.concatenate(ap), np.concatenate(al)
    ods = max([(2*np.sum((fp>=th)*fl)/(2*np.sum((fp>=th)*fl)+np.sum((fp>=th)*(1-fl))+np.sum((fp<th)*fl)+1e-8), th) for th in t])
    return {'ODS': ods[0], 'ODS_thresh': ods[1], 'OIS': np.mean(ois), 'AP': average_precision_score(fl, fp) if np.sum(fl)>0 else 0}

m = metrics(preds, gts)
print(f"\nDPED: ODS={m['ODS']:.4f} | OIS={m['OIS']:.4f} | AP={m['AP']:.4f}")

import json
with open(OUTPUT_DIR / 'dped_metrics.json', 'w') as f:
    json.dump({'model': 'DPED', 'bio': 'Hierarchical pathway + Swin', 'improvement': 'Precision and efficiency', 'metrics': m}, f, indent=2)
print("✅ DPED complete!")